# 📐 Day 04a: Normalization — 1NF to BCNF & Functional Dependencies

---

## 🎯 What You'll Master Today
- Functional dependencies & closures
- 1NF, 2NF, 3NF, BCNF step-by-step
- Lossless decomposition & dependency preservation
- Normalize a given table (interview classic)

---

```
╔══════════════════════════════════════════════════════════════╗
║              NORMAL FORMS LADDER                             ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  UNF → 1NF:  Eliminate repeating groups (atomic values)     ║
║  1NF → 2NF:  Remove partial dependencies                   ║
║               (non-key attr depends on PART of composite PK)║
║  2NF → 3NF:  Remove transitive dependencies                ║
║               (non-key depends on another non-key)          ║
║  3NF → BCNF: Every determinant is a candidate key          ║
║               (stricter version of 3NF)                     ║
║                                                              ║
║  Practical: 3NF is usually sufficient for interviews        ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================
# 1. Functional Dependency & Closure
# ============================================

def compute_closure(attrs: set, fds: list[tuple[set, set]]) -> set:
    """
    Compute attribute closure of 'attrs' under functional dependencies.
    fds: list of (lhs_set, rhs_set) tuples
    """
    closure = set(attrs)
    changed = True
    
    while changed:
        changed = False
        for lhs, rhs in fds:
            if lhs.issubset(closure) and not rhs.issubset(closure):
                closure |= rhs
                changed = True
    
    return closure


def find_candidate_keys(all_attrs: set, fds: list[tuple[set, set]]) -> list[frozenset]:
    """Find all candidate keys using closure algorithm."""
    from itertools import combinations
    
    candidate_keys = []
    
    for size in range(1, len(all_attrs) + 1):
        for combo in combinations(all_attrs, size):
            combo_set = set(combo)
            # Skip if superset of existing candidate key
            if any(ck.issubset(combo_set) for ck in candidate_keys):
                continue
            closure = compute_closure(combo_set, fds)
            if closure == all_attrs:
                candidate_keys.append(frozenset(combo_set))
    
    return candidate_keys


# Example: R(A, B, C, D, E)
# FDs: A→B, BC→D, D→E, A→C
all_attrs = {'A', 'B', 'C', 'D', 'E'}
fds = [
    ({'A'}, {'B'}),
    ({'B', 'C'}, {'D'}),
    ({'D'}, {'E'}),
    ({'A'}, {'C'}),
]

# Compute closure of {A}
closure_A = compute_closure({'A'}, fds)
print(f"  Closure of {{A}}+ = {closure_A}")
print(f"  Is {{A}} a candidate key? {closure_A == all_attrs} ✅")
print()

# Find all candidate keys
cks = find_candidate_keys(all_attrs, fds)
print(f"  Candidate Keys: {[set(ck) for ck in cks]}")

In [ ]:
# ============================================
# 2. Step-by-Step Normalization Example
# ============================================

print("  NORMALIZATION WALK-THROUGH")
print("  " + "═" * 60)
print()

# Unnormalized table
print("  ❌ UNNORMALIZED (UNF):")
print("  StudentCourse(roll, name, courses, instructor, dept, dept_head)")
print("  ┌──────┬───────┬──────────────┬────────────┬──────┬──────────┐")
print("  │ roll │ name  │ courses      │ instructor │ dept │ dept_head│")
print("  ├──────┼───────┼──────────────┼────────────┼──────┼──────────┤")
print("  │ 101  │ Alice │ {DBMS, OS}   │ {Dr.X,Dr.Y}│ CS   │ Dr.Z    │")
print("  │ 102  │ Bob   │ {DBMS}       │ {Dr.X}     │ CS   │ Dr.Z    │")
print("  └──────┴───────┴──────────────┴────────────┴──────┴──────────┘")
print("  Problem: courses & instructor are multivalued (not atomic)")
print()

print("  ✅ 1NF — Atomic values (no repeating groups):")
print("  ┌──────┬───────┬────────┬────────────┬──────┬──────────┐")
print("  │ roll │ name  │ course │ instructor │ dept │ dept_head│")
print("  ├──────┼───────┼────────┼────────────┼──────┼──────────┤")
print("  │ 101  │ Alice │ DBMS   │ Dr.X       │ CS   │ Dr.Z     │")
print("  │ 101  │ Alice │ OS     │ Dr.Y       │ CS   │ Dr.Z     │")
print("  │ 102  │ Bob   │ DBMS   │ Dr.X       │ CS   │ Dr.Z     │")
print("  └──────┴───────┴────────┴────────────┴──────┴──────────┘")
print("  PK: (roll, course)")
print("  FDs: roll→name,dept. course→instructor. dept→dept_head")
print()

In [ ]:
# ============================================
# 2NF and 3NF decomposition
# ============================================

print("  ✅ 2NF — Remove partial dependencies:")
print("  ─" * 30)
print("  PK = (roll, course)")
print("  Partial dep: roll → name, dept, dept_head (depends on PART of PK)")
print("  Partial dep: course → instructor (depends on PART of PK)")
print()
print("  Decompose into:")
print("    Student(roll PK, name, dept, dept_head)     ← from roll→name,dept,dept_head")
print("    Course(course PK, instructor)                ← from course→instructor")
print("    Enrollment(roll FK, course FK)               ← relationship table")
print()

print("  ✅ 3NF — Remove transitive dependencies:")
print("  ─" * 30)
print("  In Student: roll → dept → dept_head")
print("  Transitive: dept_head depends on dept (non-key), not directly on roll")
print()
print("  Decompose Student further:")
print("    Student(roll PK, name, dept FK)              ← remove dept_head")
print("    Department(dept PK, dept_head)                ← new table")
print()
print("  FINAL 3NF TABLES:")
print("    1. Student(roll, name, dept FK)")
print("    2. Department(dept, dept_head)")
print("    3. Course(course, instructor)")
print("    4. Enrollment(roll FK, course FK)")
print()
print("  💡 Each table: every non-key attribute depends on the key,")
print("     the whole key, and nothing but the key — so help me Codd! 😄")

In [ ]:
# ============================================
# 3. BCNF — When 3NF Isn't Enough
# ============================================

print("  BCNF vs 3NF")
print("  " + "═" * 55)
print()
print("  3NF rule: No transitive dependency of non-prime on key")
print("  BCNF rule: For EVERY FD X→Y, X must be a super key")
print()
print("  Example where 3NF ≠ BCNF:")
print("  ─" * 25)
print("  Table: StudentAdvisor(student, subject, advisor)")
print("  FDs:")
print("    (student, subject) → advisor     PK")
print("    advisor → subject                ← VIOLATION!")
print()
print("  advisor → subject: advisor is NOT a super key → NOT BCNF")
print("  But it IS 3NF (subject is a prime attribute — part of PK)")
print()
print("  BCNF Decomposition:")
print("    AdvisorSubject(advisor PK, subject)")
print("    StudentAdvisor(student, advisor FK)")
print()
print("  ⚠️ BCNF may not preserve all FDs!")
print("  💡 3NF always preserves dependencies → preferred in practice")

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | What is a functional dependency? | X→Y means X uniquely determines Y. Given X, there's exactly one Y |
| 2 | 2NF vs 3NF? | 2NF: no partial dependency on PK. 3NF: additionally no transitive dependency |
| 3 | When is BCNF needed? | When a non-candidate-key determines something. Stricter than 3NF |
| 4 | Normalize this table... | Classic interview Q. Break into 1NF→2NF→3NF step by step |
| 5 | Is denormalization bad? | No! Read-heavy systems denormalize for performance (reporting, analytics) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • 1NF: atomic values, no repeating groups               │
## │  • 2NF: no partial dependency (on part of composite PK)  │
## │  • 3NF: no transitive dependency (A→B→C, remove B→C)     │
## │  • BCNF: every determinant is a candidate key            │
## │  • "Key, whole key, nothing but the key" = 3NF mantra    │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **SQL SELECT Basics** — WHERE, ORDER BY, LIKE, CASE WHEN